# Energy Trading Briefing Agent — Colab workflow
Run these cells in order. Uses your existing NVIDIA key in **Colab Secrets**.
Wait for your synced project folder to finish syncing to Drive before starting.
This notebook runs against an explicitly selected historical date. It never relabels old data as current.
The LLM selects source statements; verification checks source fidelity, not the truth of the original source.


## 1. Mount Drive and prepare a local runtime copy


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, sys, subprocess, json

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/energy-trading-briefing-agent')
RUNTIME_PROJECT = Path('/content/energy-trading-briefing-agent')
required = ['run_pipeline.py', 'pipeline_status.py', 'agents/source_loading.py', 'check_project.py']
missing = [name for name in required if not (DRIVE_PROJECT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Upload the updated project to {DRIVE_PROJECT}. Missing: {missing}')
# Local disk supports execution locks; generated outputs are copied back to Drive.
shutil.copytree(DRIVE_PROJECT, RUNTIME_PROJECT, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('.git', '.env', '.env.*', '__pycache__',
                                             '.pytest_cache', '.venv', 'venv', 'runs'))
os.chdir(RUNTIME_PROJECT)
print('Working directory:', RUNTIME_PROJECT)


## 2. Install dependencies


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')


## 3. Load your saved key
Set `SECRET_NAME` to the name you already use in Colab Secrets. Enable notebook access for that secret.
The value stays in the runtime environment; it is not printed or written to the project.


In [ ]:
from google.colab import userdata
SECRET_NAME = 'NVIDIA_API_KEY'
os.environ.pop('NVIDIA_API_KEY', None)
try:
    key = userdata.get(SECRET_NAME)
except Exception:
    raise RuntimeError(f'Cannot access {SECRET_NAME}. Check its name and notebook access in Colab Secrets.') from None
if not key or not key.strip():
    raise RuntimeError('The saved NVIDIA key is empty.')
os.environ['NVIDIA_API_KEY'] = key.strip()
del key
print('NVIDIA key loaded securely.')


## 4. Choose a genuine data date and inspect coverage
July 13, 2026 is a historical example with saved inputs for all four domains.
To use another date, first provide its actual source data. Preflight reads saved normalized inputs without regenerating them.


In [ ]:
from datetime import date
AS_OF_DATE = '2026-07-13'
REQUIRE_COMPLETE = True
run_succeeded = False
if date.fromisoformat(AS_OF_DATE).isoformat() != AS_OF_DATE:
    raise ValueError('Use YYYY-MM-DD.')
subprocess.run([sys.executable, 'run_pipeline.py', '--date', AS_OF_DATE, '--check'], check=True)


## 4b. Check everything offline before spending API credits
This repeats the local readiness checks using Colab's installed dependencies. The rehearsal uses real saved inputs with a deterministic test substitute for the LLM. It is not evidence of live model quality.


In [ ]:
subprocess.run([sys.executable, 'check_project.py', '--date', AS_OF_DATE, '--rehearse'], check=True)
review_folder = RUNTIME_PROJECT / 'output/readiness' / AS_OF_DATE
shutil.copytree(review_folder, DRIVE_PROJECT / 'output/readiness' / AS_OF_DATE, dirs_exist_ok=True)
from IPython.display import Markdown, display
display(Markdown((review_folder / 'readiness.md').read_text(encoding='utf-8')))


## 5. Generate the brief, verification report, and PDF
This cell makes a real NVIDIA API call. Rerunning it regenerates the selected date.
Failures stop the workflow. Existing artifacts are not presented as newly generated results.
The local CLI browser option `--open` is not used in Colab.


In [ ]:
run_succeeded = False
command = [sys.executable, 'run_pipeline.py', '--date', AS_OF_DATE]
if REQUIRE_COMPLETE:
    command.append('--require-complete')
try:
    subprocess.run(command, check=True)
    run_succeeded = True
finally:
    # Preserve run diagnostics even if generation fails, without copying credentials.
    (DRIVE_PROJECT / 'runs').mkdir(exist_ok=True)
    for record in Path('runs').glob(f'{AS_OF_DATE}_*.json'):
        shutil.copy2(record, DRIVE_PROJECT / 'runs' / record.name)
if run_succeeded:
    (DRIVE_PROJECT / 'briefs').mkdir(exist_ok=True)
    for suffix in ('.md', '_verified.md', '_verified.pdf', '_verification_report.json', '_prompt_and_response.json'):
        artifact = Path('briefs') / f'{AS_OF_DATE}{suffix}'
        if not artifact.is_file():
            run_succeeded = False
            raise FileNotFoundError(f'Expected output missing: {artifact}')
        shutil.copy2(artifact, DRIVE_PROJECT / 'briefs' / artifact.name)
    print('Brief, PDF, audit, and run records saved to Google Drive.')


## 6. Read the generated brief and inspect the audit


In [ ]:
from IPython.display import Markdown, display
import pandas as pd
if not run_succeeded:
    raise RuntimeError('Complete the generation cell successfully first.')
report = json.loads(Path(f'briefs/{AS_OF_DATE}_verification_report.json').read_text(encoding='utf-8'))
run_id = report['run_id']
record = json.loads(Path(f'runs/{AS_OF_DATE}_{run_id}.json').read_text(encoding='utf-8'))
if record['status'] != 'success':
    raise RuntimeError('The corresponding run did not succeed.')
display(Markdown(Path(f'briefs/{AS_OF_DATE}_verified.md').read_text(encoding='utf-8')))
display(pd.DataFrame(report['coverage']))
display(pd.DataFrame(record['stages']))
print(json.dumps(report['summary'], indent=2))
print('A grounding score measures draft statements matched to sources, not independent factual accuracy.')


## 7. Download the PDF


In [ ]:
from google.colab import files
if not run_succeeded:
    raise RuntimeError('Complete generation first.')
files.download(str(RUNTIME_PROJECT / 'briefs' / f'{AS_OF_DATE}_verified.pdf'))


## 8. Run isolated offline regression tests (optional)


In [ ]:
tests = [
    'eval/test_price_agent.py', 'eval/test_weather_agent.py', 'eval/test_outage_agent.py',
    'eval/test_regulatory_agent.py', 'eval/test_cross_referencing_agent.py',
    'eval/test_verifier_agent.py', 'eval/test_regressions.py', 'eval/test_pipeline_status.py', 'eval/test_readiness.py',
]
subprocess.run([sys.executable, '-m', 'pytest', *tests, '-q'], check=True)


## What remains outside this notebook
- Fresh source collection is separate; historical inputs do not establish current market conditions.
- The regression suite does not establish real-world usefulness or accuracy; human review is still needed.
- No emails are sent. Outputs and diagnostics are saved to Drive and the PDF can be downloaded.
- For updated code, wait for Drive sync to finish and restart this workflow in a fresh Colab runtime.
